In [1]:
import sys
import os
import pandas as pd

# Add the parent directory of 'app' to the Python path
sys.path.append(os.path.abspath('D:/GitHub/personal_finances_analysis'))

from src.ofx_dataprep import OfxDataPrep
from src.llm_finance import LLMFinance

# In root path we have the folders input, output and processed
root_path = "D:\GitHub\_DADOS\personal_finances"
input_path = os.path.join(root_path, "input")

Pre-processing and Saving Data
- bank account 
- credit card transactions

In [2]:
dataprep = OfxDataPrep()
transactions = dataprep.read_and_prep_data(data_path=input_path)

transactions.to_csv(os.path.join(root_path, "output", "transactions_prep.csv"), index=False, encoding='utf-8')

Read processed data

In [3]:
transactions = pd.read_csv(os.path.join(root_path, "output", "transactions_prep.csv"), encoding='utf-8')
transactions.shape

(639, 12)

### Bank Account Transactions
- is_installment = False = There isn't transactions with installment = True
- Is not good idea control how much you received trought bank "in" transactions because you schedulled a transaction of R$4k from Santander to Nubank, and everytime the value is lower or higher the scheduled value. So in this way you need to check your SANTANDER account, every month, to collect this information and fill the spreadsheet
- In transactions doesn't have valuable information for the analysis, for now I will concentrade in the rest of transactions
- It's possible to identify some bills and create an automation to collect this data, but I don't sure if this effort will save time because every bill that exists in BANK OUT transactions I have to pay manually, so I could fill the spreadsheet while I'm paying the bill.
- bank account out transactions has a lot of pix transactions.

In [21]:
df_bank = transactions.query("type == 'Bank'").copy()
display(df_bank.shape)

df_bank_in = df_bank.query("in_out == 'in'").copy()
df_bank_out = df_bank.query("in_out == 'out'").copy()

print(df_bank_in.shape, df_bank_out.shape)

print(df_bank.columns)

(160, 12)

(52, 12) (108, 12)
Index(['type', 'date', 'amount', 'id', 'memo', 'account_id', 'number',
       'institution', 'year', 'year_month', 'in_out', 'is_installment'],
      dtype='object')


In [ ]:
# BANK IN
# df_bank_in.groupby('memo')['amount'].sum().sort_values(ascending=False)

BANK OUT

In [44]:
memo_not_filter = ["Pagamento de boleto efetuado - CONDOMINIO EDIFICIO FATTO SANTO ANDRE",
                   'Pagamento de boleto efetuado - PJBANK PAGAMENTOS S A','Transferência enviada pelo Pix - Pjbank Pagamentos S A - 18.191.228/0001-71 - SUPERLOGICA SCD S.A. (0481) Agência: 1 Conta: 17596-0',
                   'Aplicação RDB',
                   'Transferência enviada pelo Pix - MATHEUS GONZALEZ EUGENIO - •••.210.918-•• - BCO SANTANDER (BRASIL) S.A. (0033) Agência: 218 Conta: 1015097-6', # These are the wrong transfers scheduled
                   'Transferência enviada pelo Pix - Claro - 40.432.544/0001-47 - CLARO PAY S.A. IP Agência: 1 Conta: 1872704-2','Pagamento de boleto efetuado - Claro',
                   'Saque',
                   'Recarga de celular',
                   'Pagamento de boleto efetuado - ELETROPAULO','Pagamento de boleto efetuado - ELETROPAULO METR ELETR SP SA',
                   'Pagamento de boleto efetuado - ELETROPAULO METROPOLITANA','Transferência enviada pelo Pix - ENEL DISTRIBUICAO SAO PAULO - 61.695.227/0001-93 - ITAÚ UNIBANCO S.A. (0341) Agência: 911 Conta: 4486-5',
                   ]

# BANK OUT
#df_bank_out.query(f'memo not in {memo_not_filter}').groupby('memo')['amount'].sum().sort_values(ascending=True)

In [45]:
memo_filter = ['Compra no débito - Autopass S.A.','Compra no débito - AUTOPASS S.A.',
               'Transferência enviada pelo Pix - ON TECNOLOGIA DE MOBILIDADE URBANA LTDA. - 26.054.490/0001-00 - BCO BRADESCO S.A. (0237) Agência: 127 Conta: 29346-6']

#display(df_bank_out.query(f"memo in {memo_filter}")[['type', 'date', 'amount', 'id', 'memo', 'account_id', 'number','institution', 'year', 'year_month', 'in_out', 'is_installment']].sort_values(by='year_month', ascending=True))
#display(df_bank_out.query(f"memo in {memo_filter}").groupby(['memo','year_month'])['amount'].sum().reset_index().sort_values(by='year_month', ascending=True))


### Credit Card Transactions

- Credit card transactions means devolution of some transactions or payment of monthly credit card amount. It's not necessary monitoring it.

In [4]:
df_cc = transactions.query('type == "Credit Card"').copy()
print(df_cc.shape, df_cc.columns)

df_cc_in = df_cc.query("in_out == 'in'").copy()
df_cc_out = df_cc.query("in_out == 'out'").copy()
print(df_cc_in.shape, df_cc_out.shape)

(479, 12) Index(['type', 'date', 'amount', 'id', 'memo', 'account_id', 'number',
       'institution', 'year', 'year_month', 'in_out', 'is_installment'],
      dtype='object')
(18, 12) (461, 12)


In [7]:
df_cc_in.groupby('memo')['amount'].sum().sort_values(ascending=False)

memo
Pagamento recebido                     28987.53
CrÃ©dito de atraso                      9933.68
Estorno de "Bp"                          592.48
Encerramento de dÃ­vida                  246.60
Estorno de "Pg *Celetus Cltusclick"       57.00
Estorno de "Riachuelo Lo*Riachuelo"       36.76
Name: amount, dtype: float64

In [14]:
df_cc_out.groupby('year_month')['amount'].sum().sort_values(ascending=False)

year_month
2025-07     -590.83
2025-05    -3569.95
2025-04    -3579.86
2024-12    -3725.09
2025-01    -4182.86
2025-06    -4393.23
2025-02   -10300.00
2025-03   -10640.65
Name: amount, dtype: float64

In [20]:
df_cc_out.query('year_month == "2025-06"').sort_values(by='date', ascending=True).groupby(['year_month', 'memo'])['amount'].sum().reset_index().to_csv(os.path.join(root_path, "output", "cc_out_2025-06.csv"), index=False, encoding='utf-8')

In [10]:
df_cc_out.dtypes

type               object
date               object
amount            float64
id                 object
memo               object
account_id         object
number             object
institution        object
year                int64
year_month         object
in_out             object
is_installment       bool
dtype: object

### Summary of Actions


Transactions that I want to view month by month
- BANK OUT - 'Recarga de celular'